# JobSpy Playground 🔎

Интерактивный ноутбук для тестирования функционала **JobSpy** и просмотра вакансий
**с полными описаниями и всей доп-информацией**.

**Как пользоваться:** меняй блок `CONFIG` ниже → `Run All` (или по очереди Shift+Enter).
Карточки вакансий рендерятся с описанием, зарплатой, компанией, ссылками и т.д.

> Запускать из venv проекта: ядро Python должно видеть установленный `jobspy`.


In [1]:
# --- Imports & настройки отображения ---
import textwrap
import pandas as pd
from IPython.display import display, HTML, Markdown

from jobspy import scrape_jobs
from jobspy.model import Country, JobType, Site

pd.set_option("display.max_colwidth", None)   # не обрезать длинный текст
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print("Площадки:", [s.value for s in Site])
print("Типы занятости:", sorted({jt.value[0] for jt in JobType}))
print("Стран поддерживается (Indeed/Glassdoor):",
      len([c for c in Country if c not in (Country.US_CANADA, Country.WORLDWIDE)]))


Площадки: ['linkedin', 'indeed', 'zip_recruiter', 'glassdoor', 'google', 'bayt', 'naukri', 'bdjobs']
Типы занятости: ['contract', 'fulltime', 'internship', 'nights', 'other', 'parttime', 'perdiem', 'summer', 'temporary', 'volunteer']
Стран поддерживается (Indeed/Glassdoor): 72


## 1. Настрой поиск

`fetch_description=True` обязателен, чтобы у LinkedIn появились полные описания и прямые ссылки.
Indeed отдаёт описания сразу. Подробности параметров — в `jobspy_cli.py --help`.


In [2]:
CONFIG = dict(
    site_name=["indeed", "linkedin"],   # linkedin, indeed, glassdoor, google, zip_recruiter, bayt, naukri, bdjobs
    search_term="Business Analyst",      # "" = любые роли
    google_search_term="Business Analyst jobs in Germany since last week",
    location="Germany",                  # напр. "Heidelberg, Germany"
    country_indeed="germany",            # нужно для Indeed/Glassdoor
    distance=50,                          # радиус, мили
    results_wanted=15,                    # на площадку
    hours_old=72,                         # за последние N часов
    is_remote=False,                      # True = только удалёнка
    job_type=None,                        # fulltime / parttime / internship / contract / temporary
    easy_apply=None,
    linkedin_fetch_description=True,      # ← полные описания + прямые ссылки LinkedIn
    description_format="markdown",        # markdown / html / plain
    enforce_annual_salary=False,
    verbose=1,
)
CONFIG


{'site_name': ['indeed', 'linkedin'],
 'search_term': 'Business Analyst',
 'google_search_term': 'Business Analyst jobs in Germany since last week',
 'location': 'Germany',
 'country_indeed': 'germany',
 'distance': 50,
 'results_wanted': 15,
 'hours_old': 72,
 'is_remote': False,
 'job_type': None,
 'easy_apply': None,
 'linkedin_fetch_description': True,
 'description_format': 'markdown',
 'enforce_annual_salary': False,
 'verbose': 1}

## 2. Запусти скрейпинг

In [3]:
jobs = scrape_jobs(**CONFIG)
print(f"Найдено вакансий: {len(jobs)}")
jobs["site"].value_counts() if len(jobs) else "Пусто — ослабь hours_old / поменяй location / убери площадку"


Найдено вакансий: 30


site
indeed      15
linkedin    15
Name: count, dtype: int64

## 3. Обзор результатов — компактная таблица

In [4]:
SUMMARY_COLS = ["site", "title", "company", "location", "date_posted",
                "job_type", "is_remote", "min_amount", "max_amount", "currency", "job_url"]
summary = jobs[[c for c in SUMMARY_COLS if c in jobs.columns]].copy()
print(f"{len(jobs)} вакансий | колонок всего: {len(jobs.columns)}")
print("Доступные колонки:", list(jobs.columns))
summary


30 вакансий | колонок всего: 34
Доступные колонки: ['id', 'site', 'job_url', 'job_url_direct', 'title', 'company', 'location', 'date_posted', 'job_type', 'salary_source', 'interval', 'min_amount', 'max_amount', 'currency', 'is_remote', 'job_level', 'job_function', 'listing_type', 'emails', 'description', 'company_industry', 'company_url', 'company_logo', 'company_url_direct', 'company_addresses', 'company_num_employees', 'company_revenue', 'company_description', 'skills', 'experience_range', 'company_rating', 'company_reviews_count', 'vacancy_count', 'work_from_home_type']


,site,title,company,location,date_posted,job_type,is_remote,min_amount,max_amount,currency,job_url
0,indeed,Junior ABAP Developer (w/m/d),Aareal Bank,"Wiesbaden, HE, DE",2026-06-11,NaN,False,None,None,None,https://de.indeed.com/viewjob?jk=a9f0797444582dcb
1,indeed,Data Analyst als Experte Fachkonzepte & Data Lake (m/w/d),Zurich Insurance,"Frankfurt am Main, HE, DE",2026-06-11,"parttime, fulltime",False,None,None,None,https://de.indeed.com/viewjob?jk=a893707c9d4a9f8f
2,indeed,Senior Manager - Change Management,Kuoni Tumlare,"Frankfurt am Main, HE, DE",2026-06-11,fulltime,False,None,None,None,https://de.indeed.com/viewjob?jk=58ab63ab0f0ca4de
3,indeed,Technical Product Owner im Bereich Tax Technology & E-Invoicing (m/w/d),cbs Corporate Business Solutions,"Heidelberg, BW, DE",2026-06-11,fulltime,False,None,None,None,https://de.indeed.com/viewjob?jk=264e43c4807fca7d
4,indeed,Technical Product Owner im Bereich Tax Technology & E-Invoicing (m/w/d),cbs Corporate Business Solutions,"Dortmund, NW, DE",2026-06-11,fulltime,False,None,None,None,https://de.indeed.com/viewjob?jk=187fb3235143fbf5
5,indeed,Business Analyst Legal Operations (m/w/d),REWE Group,"Köln, NW, DE",2026-06-11,fulltime,True,None,None,None,https://de.indeed.com/viewjob?jk=8af2b590ed5c32bc
6,indeed,Salesforce AI & Analytics Analyst (all genders),Accenture,"Düsseldorf, NW, DE",2026-06-11,fulltime,True,None,None,None,https://de.indeed.com/viewjob?jk=2487537631233563
7,indeed,Salesforce AI & Analytics Analyst (all genders),Accenture,"Stuttgart, BW, DE",2026-06-11,fulltime,True,None,None,None,https://de.indeed.com/viewjob?jk=97f0444523bcd251
8,indeed,Salesforce AI & Analytics Analyst (all genders),Accenture,"München, BY, DE",2026-06-11,fulltime,True,None,None,None,https://de.indeed.com/viewjob?jk=bd4b31c8d9bf7c6b
9,indeed,Salesforce AI & Analytics Analyst (all genders),Accenture,"Kronberg, HE, DE",2026-06-11,fulltime,True,None,None,None,https://de.indeed.com/viewjob?jk=b6a2f2b6569ff293


## 4. Полные карточки вакансий (с описанием и всей доп-информацией)

Функция `show_jobs()` рендерит каждую вакансию как карточку: все непустые поля + логотип +
описание (Markdown). `show_jobs(jobs, n=5)` — первые 5; `show_jobs(jobs.tail(3))` и т.п.


In [5]:
# Человекочитаемые подписи полей
FIELD_LABELS = {
    "company": "Компания", "location": "Локация", "date_posted": "Опубликовано",
    "is_remote": "Удалёнка", "job_type": "Тип занятости", "job_level": "Уровень",
    "job_function": "Функция", "company_industry": "Индустрия",
    "interval": "Период оплаты", "min_amount": "Зарплата от", "max_amount": "Зарплата до",
    "currency": "Валюта", "salary_source": "Источник зарплаты",
    "company_url": "Сайт компании (board)", "company_url_direct": "Сайт компании (прямой)",
    "job_url": "Ссылка (board)", "job_url_direct": "Ссылка (прямая)",
    "emails": "Email", "listing_type": "Тип листинга",
    "company_addresses": "Адрес", "company_num_employees": "Сотрудников",
    "company_revenue": "Выручка", "company_description": "О компании",
    "skills": "Навыки", "experience_range": "Опыт",
    "company_rating": "Рейтинг компании", "company_reviews_count": "Отзывов",
    "vacancy_count": "Вакансий", "work_from_home_type": "Формат работы",
}
# Поля, которые НЕ кладём в таблицу метаданных (выводим отдельно / слишком длинные)
SKIP_IN_TABLE = {"id", "site", "title", "description", "company_logo",
                 "banner_photo_url", "company_name"}

def _is_empty(v):
    return v is None or (isinstance(v, float) and pd.isna(v)) or (isinstance(v, str) and not v.strip())

def _fmt_value(field, v):
    if field in ("job_url", "job_url_direct", "company_url", "company_url_direct") and isinstance(v, str) and v.startswith("http"):
        return f'<a href="{v}" target="_blank">{v}</a>'
    if field == "emails" and isinstance(v, str):
        return ", ".join(f'<a href="mailto:{e.strip()}">{e.strip()}</a>' for e in v.split(","))
    return str(v)

def render_job(row):
    title = row.get("title", "—")
    site = row.get("site", "")
    url = row.get("job_url", "")
    logo = row.get("company_logo")
    logo_html = (f'<img src="{logo}" style="height:42px;float:right;border-radius:6px">'
                 if isinstance(logo, str) and logo.startswith("http") else "")
    header = (f'<div style="border:1px solid #ddd;border-radius:10px;padding:14px 16px;margin:14px 0 0 0;'
              f'background:#fafafa">{logo_html}'
              f'<div style="font-size:1.15em;font-weight:700">'
              f'<a href="{url}" target="_blank" style="text-decoration:none">{title}</a></div>'
              f'<div style="color:#888;font-size:0.85em;margin-top:2px">via {site}</div>')
    rows_html = ""
    for field in row.index:
        if field in SKIP_IN_TABLE:
            continue
        v = row[field]
        if _is_empty(v):
            continue
        label = FIELD_LABELS.get(field, field)
        rows_html += (f'<tr><td style="padding:3px 10px 3px 0;color:#555;vertical-align:top;'
                      f'white-space:nowrap"><b>{label}</b></td>'
                      f'<td style="padding:3px 0">{_fmt_value(field, v)}</td></tr>')
    table = f'<table style="border-collapse:collapse;margin-top:8px;font-size:0.9em">{rows_html}</table></div>'
    return header + table

def show_jobs(df, n=5):
    """Карточка + описание (Markdown) для первых n строк df."""
    if len(df) == 0:
        print("Нет вакансий для показа."); return
    for _, row in df.head(n).iterrows():
        display(HTML(render_job(row)))
        desc = row.get("description")
        if isinstance(desc, str) and desc.strip():
            display(Markdown("**📝 Описание:**\n\n" + desc))
        else:
            display(Markdown("_(описание недоступно — для LinkedIn включи `linkedin_fetch_description=True`)_"))
        display(HTML('<hr style="border:none;border-top:2px dashed #ccc;margin:18px 0">'))

print("Готово. Используй: show_jobs(jobs, n=5)")


Готово. Используй: show_jobs(jobs, n=5)


In [6]:
# Первые 5 вакансий целиком, с описаниями
show_jobs(jobs, n=5)


Ссылка (board),https://de.indeed.com/viewjob?jk=a9f0797444582dcb
Ссылка (прямая),https://www.aareal-bank.com/karriere/jobs-bewerbung/berufserfahren?tx_aarealjobs_hrportallistshow%5Baction%5D=show&tx_aarealjobs_hrportallistshow%5Bcontroller%5D=Job&tx_aarealjobs_hrportallistshow%5Bjob%5D=732&cHash=15afec34b03d71eb85f6d0f32324f5f5
Компания,Aareal Bank
Локация,"Wiesbaden, HE, DE"
Опубликовано,2026-06-11
Удалёнка,False
Сайт компании (board),https://de.indeed.com/cmp/Aareal-Bank
Сайт компании (прямой),https://www.aareal-bank.com
Адрес,"Wiesbaden, GM"
Сотрудников,"1,001 to 5,000"
Выручка,$500M to $1B (USD)


**📝 Описание:**

Für diese Position suchen wir eine ideenreiche und innovative Persönlichkeit, die dank ihrer exzellenten Analysefähigkeiten das Team mit Fokus auf Bilanzierungslösungen verstärkt. Da wir hierbei weitgehend auf unsere SAP-basierten Eigenentwicklungen zurückgreifen, sind keine spezifischen SAP-Modulkenntnisse erforderlich. Wenn du hingegen über einen ausgeprägten Teamgeist verfügst und den Dingen gerne durch konstruktives und kritisches Nachfragen auf den Grund gehst, sagen wir: Herzlich willkommen bei der Aareal Bank Group.

#### **Darauf kannst du dich freuen:**

* Als Junior ABAP Entwickler (w/m/d) verstärkst Du das Team rund um die Weiterentwicklung und Wartung einer komplexen Bilanzierungslösung unter SAP S/4HANA – eine Eigenentwicklung, die tief in die Finanzprozesse der Bank integriert ist.
* Die Aareal Bank bietet ihren Kunden strukturierte Finanzierungen. Die bilanzielle Abbildung solcher Geschäfte ist rundum anspruchsvoll und sorgt sicher dafür, dass keine Langeweile aufkommt.
* Intensive Einarbeitung durch ein Team von erfahrenen Kolleginnen und Kollegen in alle relevanten Aspekte der Bilanzierung, dazu gehören insbesondere die Klassifizierung und Bewertung von Finanzinstrumenten, Bildung der Risikovorsorge sowie Hedge Accounting.
* Deine Einarbeitung wird individuell gestaltet und erfolgt in einem interdisziplinären Team mit Entwicklern, Business-Analysten und Fachbereichen. Sie umfasst sowohl die konkrete technische Implementierung als auch die Vermittlung fachlicher Grundlagen zu den geltenden Rechnungslegungsvorschriften.
* Über die reinen Wartungs- und Entwicklungsarbeiten hinaus erstellst Du auch technische Konzepte sowie Betriebslösungen, die Stabilität, Performance und Wartbarkeit sicherstellen.

#### **Darauf können wir uns freuen:**

* Ein abgeschlossenes Studium der Informatik, Wirtschaftsinformatik, Mathematik, Wirtschaftsmathematik oder Wirtschaftswissenschaften mit technischem Schwerpunkt - oder eine vergleichbare Qualifikation
* Nachweisbar erfahren im Umgang mit SAP ABAP OO und ABAP 4HANA. Hilfreich ist darüber hinaus Erfahrung in CDS-Views und OData.
* Im Rahmen einer Praktikums- oder Werkstudententätigkeit hast du erste Erfahrung in Unternehmen in der Wartung und Entwicklung im SAP-Umfeld gesammelt. Natürlich bist Du auch mit darüber hinausgehender Erfahrung bei uns willkommen.
* Idealerweise Routine mit agilen Vorgehensmodellen – z. B. SCRUM oder Kanban
* Nice to have: Kenntnisse im Finanzwesen und Bilanzierung sowie Modulkenntnisse in SAP FI/CO
* Fließende Deutschkenntnisse in Wort und Schrift

**Das machen wir aanders!**
---------------------------

* ### **Aabwechslungsreiche Aufgaben**

  Bei uns kannst du deine Expertise einbringen, Themen voranbringen und Neues gestalten. Wir haben eine Kultur, in der sich alle aktiv einbringen und Einfluss nehmen können.
* ### **Aaußergewöhnliche Weiterentwicklung**

  Wir fördern jede Person gleichermaßen, deshalb unterstützen wir dich mit individuellen Entwicklungsmöglichkeiten und Lernwegen. Damit du bestehende Kompetenzen ausbauen und neue zukunftsweisende Fähigkeiten aufbauen kannst.
* ### **Aausgeglichene Work-Life Balance**

  Mit mobilem Arbeiten und flexiblen Arbeitszeitmodellen hast du die Möglichkeit, deine Arbeit individuell zu gestalten. So lassen sich Arbeit, Familie und die persönliche Lebenssituation bestmöglich unter einen Hut bringen.
* ### **Aausgezeichnete Benefits**

  Unsere Services und Benefits bieten dir einen echten Mehrwert, die uns so zu einem Top-Arbeitgeber machen. Bei uns erhältst du beispielsweise Vergünstigungen bei Kooperationspartnern oder Zuschüsse zum Nahverkehr. Ebenso sorgen wir mit einer betrieblichen Altersversorgung oder vermögenswirksamen Leistungen für deine Zukunft vor.
* ### **Aaktives Gesundheitsmanagement**

  Ärztliche Check-ups, Beratungsangebote, gemeinsame Sportevents, unsere beliebte Kantine oder Kaffeebar – Bei uns kannst du das nutzen, was deine Gesundheit fördert und dir guttut.

**Willkommen in der Welt der Aareal Bank!**
-------------------------------------------

Wir sind Mittelständler und Global Player. Als internationaler Immobilienspezialist arbeiten wir alle auf ein gemeinsames Ziel hin: den Erfolg unserer Kunden. Bei uns arbeiten über 1.000 Menschen aus 44 Nationen auf drei Kontinenten daran, dass wir als Bank ganz schön aanders sind! Wir sind verantwortungsvoller und erstklassiger Ausbildungsbetrieb und eine Organisation mit viel Expertise und kurzen Wegen, viel Gestaltungsspielraum und dem Anspruch, für unsere Kunden immer besser zu werden. Das macht uns stark und vor allem: aanders!

Ссылка (board),https://de.indeed.com/viewjob?jk=a893707c9d4a9f8f
Ссылка (прямая),https://www.careers.zurich.com/job/Frankfurt-Data-Analyst-als-Experte-Fachkonzepte-&-Data-Lake-%28mwd%29/1363513257/
Компания,Zurich Insurance
Локация,"Frankfurt am Main, HE, DE"
Опубликовано,2026-06-11
Тип занятости,"parttime, fulltime"
Удалёнка,False
Email,k.schmidt@zurich.com
Сайт компании (board),https://de.indeed.com/cmp/Zurich-Insurance-9
Сайт компании (прямой),https://www.zurich.com/
Адрес,Zürich


**📝 Описание:**

**Bist du bereit,** **mit dem****Besten zu rechnen****?**

  

Bei Zurich leben wir Versicherung neu. Um die Wünsche unserer Kunden noch besser zu erfüllen, gehen wir neue Wege, denken kreativ, arbeiten agil. Unsere Unternehmenskultur schenkt dir in jeder Hinsicht mehr Flexibilität – und viel Freiraum, dich optimal zu entfalten. Think big – und gerne auch international! Denn deine Entwicklungsmöglichkeiten kennen bei uns keine Grenzen. Bereit, die Zukunft in die Hand zu nehmen?

  

**Das erwartet dich**

  

* In deiner Rolle bist du am Standort Frankfurt zuständig für die Erstellung und Pflege von Fachkonzepten auf Basis der Reporting-Anforderungen aus dem Fachbereich und begleitest deren fachliche und technische Umsetzung
* Mit Freude entwickelst du das Datenmodell sowie die Layer des Datenpools CIG im Data Lake weiter und unterstützt aktiv bei der Konzeption und technischen Begleitung des Data Lake ZGD
* Du unterstützt aktiv bei der Weiterentwicklung unserer Datenarchitektur (z.B. CIG Data Domains) und bringst deine Expertise in ressortspezifische Projekte ein
* Du stehst in enger Zusammenarbeit mit verschiedenen Fachbereichen, spartenspezifischen Business Analysten:innen und weiteren Stakeholder:innen und stellst eine effiziente und zielgerichtete Umsetzung von Reporting- und Datenanforderungen sicher
* Innerhalb des Teams arbeitest Du eng zusammen mit spezialisierten Data Engineers, Data Scientisten und Data Analysten.
* Bei uns bekommst du die Möglichkeit, die CIG-Datenstrategie, moderne Datenstrukturen und zukunftsorientierte Lösungen im Commercial-Umfeld nachhaltig mitzugestalten

  

**Das bringst du mit**

  

* Strukturierte und lösungsorientierte Arbeitsweise kombiniert mit hoher Service- und Kundenorientierung
* Ausgeprägte Teamfähigkeit, Kommunikationsstärke sowie Erfahrung in der standortübergreifenden Zusammenarbeit mit unterschiedlichen Stakeholder:innen
* Erfolgreich abgeschlossenes Studium in Betriebswirtschaftslehre, Informatik, Mathematik, Data Science oder eine vergleichbare Ausbildung
* Erfahrung in der Konzeption von Fachkonzepten und Datenmodellen sowie gutes Verständnis von Reporting- und Datenarchitekturen
* Gute Kenntnisse der Commercial-Systemlandschaft inklusive Produkt- und Systemverständnis sowie fließende Deutsch- und gute Englischkenntnisse

  

**Das bieten wir dir**

  

* Flexible Arbeitsformen für eine optimale Balance zwischen Arbeiten am Arbeitsplatz und von zu Hause aus
* Zahlreiche Trainings- und Lernangebote, um fit für den Arbeitsalltag zu bleiben und eine individuelle Weiterentwicklung zu ermöglichen
* Unterstützung bei deiner Karriereplanung - von der Potenzialanalyse bis hin zur individuellen Förderung
* Eine moderne, offene und transparente Arbeitsumgebung, die Kreativität und teamübergreifende Zusammenarbeit fördert
* Ob Fitness in der Gruppe vor Ort oder virtuelles Achtsamkeitstraining von zu Hause: Unser vielfältiges Gesundheitsangebot unterstützt eine gesunde Work‑Life‑Balance
* Bikeleasing, vergünstigtes Deutschlandticket, vermögenswirksame Leistungen, ein Mitarbeitenden‑Aktienprogramm und weitere tolle Benefits

  

**Z****usätzliche Informationen**

  

Die Zurich Gruppe Deutschland unterstützt in Abhängigkeit zur individuellen Stellenanforderung und der persönlichen Eignung grundsätzlich sowohl flexible als auch Teilzeit-Arbeitsformen. Wenn du Interesse an dieser Stelle hast und deine persönliche Situation nicht ganz mit dem angegebenen Pensum (Voll- und Teilzeit) zusammenpasst, wende dich gerne an uns (k.schmidt@zurich.com), um die potenziellen Möglichkeiten zu besprechen.

  

* Arbeitsort: Frankfurt
* Teilzeit oder Vollzeit: Vollzeit

  

**Kim Schmidt** freut sich auf deine Bewerbungsunterlagen mit Angabe deiner Gehaltsvorstellungen und des möglichen Starttermins über das Karriereportal (“Jetzt Bewerben”-Button). Leider können wir Bewerbungen per E-Mail nicht berücksichtigen.

  

**Willkommen in deiner Zukunft**

  

55.000 Kolleginnen und Kollegen in 215 Ländern und Gebieten. Gemeinsam auf dem Weg, Versicherung jeden Tag neu zu denken – und die Zukunft für alle zu gestalten. Willkommen in einer starken Gemeinschaft, in der du von Anfang an etwas bewegst. Gemeinsam entwickeln wir uns weiter!

  

https://www.zurich.de/de-de/ueber-uns/ihre-karriere

Ссылка (board),https://de.indeed.com/viewjob?jk=58ab63ab0f0ca4de
Ссылка (прямая),https://careers.kuonitumlare.com/l/en/o/senior-manager-change-management?source=Indeed
Компания,Kuoni Tumlare
Локация,"Frankfurt am Main, HE, DE"
Опубликовано,2026-06-11
Тип занятости,fulltime
Удалёнка,False
Индустрия,Restaurants Travel And Leisure
Сайт компании (board),https://de.indeed.com/cmp/Kuoni-Tumlare
Сайт компании (прямой),https://www.kuonitumlare.com/
Адрес,Zürich


**📝 Описание:**

**About Us:**

At Kuoni Tumlare, we design and deliver exceptional travel experiences through a comprehensive

portfolio of destination management solutions. With over 100 years of experience, we serve a global

network of partners by offering series tours, educational trips, MICE events, and more.

For a new platform that will replace our 4 legacy enterprise systems we are looking for Business Analysts

to join our Agile teams.

**The Job:**

We are seeking a Senior Manager – Change Management to lead and drive change initiatives supporting the implementation of a new enterprise system across our global travel organization. This role will be responsible for designing and executing end-to-end change strategies that ensure successful adoption, minimal disruption, and sustained business impact.

The role requires close collaboration with business leaders, project teams, regional stakeholders, and functional experts to embed change effectively across diverse markets and cultures. A key aspect of this role is leveraging AI tools and insights to accelerate change adoption, enhance engagement, and improve decision-making.

This role is a 2 year contract, possibly extending to 3 years.

**Key Responsibilities**

**Change Strategy & Planning**

* Develop and execute a comprehensive change management strategy aligned with project goals and business priorities
* Define success metrics, adoption KPIs, and change governance frameworks
* Integrate change plans with project timelines and implementation milestones

**Stakeholder Management**

* Conduct stakeholder analysis to identify impacted groups, influence levels, and engagement strategies
* Build strong relationships with senior leaders, regional teams, and functional stakeholders
* Coach leaders to effectively sponsor and role-model change

**Change Impact & Readiness**

* Assess change impacts across processes, roles, systems, and behaviors
* Conduct organizational readiness assessments and define mitigation actions
* Identify resistance points and design targeted interventions

**Communication & Engagement**

* Design and deliver a structured communication strategy across global audiences
* Develop compelling messaging tailored to different stakeholder groups
* Partner with Global Head - Corporate Communications, Kuoni Tumlare to ensure consistency and reach

**Training & Capability Building**

* Define training strategy, curriculum, and learning journeys aligned to system implementation
* Coordinate with Head of Training DMC Europe, SMEs, and regional leads to deliver training programs
* Ensure training effectiveness through feedback loops and adoption metrics

**Adoption & Sustainment**

* Monitor adoption through data insights, user feedback, and performance metrics
* Design reinforcement mechanisms including nudges, leadership engagement, and recognition
* Ensure sustained behavioral change post-implementation

**AI-Enabled Change Acceleration**

* Advise on and deploy AI tools to enhance change activities (e.g., personalized communications, chatbots for user support, sentiment analysis, learning recommendations)
* Use AI-driven insights to track engagement, predict resistance, and tailor interventions
* Partner with digital/IT teams to integrate AI capabilities into the change ecosystem

**Key Requirements**

* English – fluent (written & spoken)
* 15+ years of experience in change management, organizational development, or transformation roles
* Proven experience leading large-scale system implementations (preferably in global or matrixed environments)
* Strong expertise in change methodologies (e.g., Prosci, Kotter, ADKAR)
* Demonstrated ability to manage stakeholders across cultures and geographies
* Experience designing and delivering communication and training strategies at scale
* Familiarity with AI tools and digital platforms that support change and learning
* Excellent facilitation, storytelling, and influencing skills

**Preferred Qualifications**

* Experience in the travel, hospitality, or service industry
* Certification in change management methodologies
* Exposure to AI-enabled transformation initiatives
* Ability to work in fast-paced, ambiguous environments

**What we offer:**

* Global Brand: Opportunity to work in an international environment
* Stability: 100 Years at the top of our field and still pushing into new territory
* Progression: We reward high performers and look to promote key talent internally
* Learning and Development opportunities for growth and Upskilling
* A Supportive Management Culture and autonomous working environment
* Company Wide Bonus Scheme
* Dedicated Employee Engagement Activities
* Flexible & Hybrid Working
* Annual Awards and Recognition for high Performers
* Friendly and Collaborative work environment

Ссылка (board),https://de.indeed.com/viewjob?jk=264e43c4807fca7d
Ссылка (прямая),https://cbsconsulting.recruitee.com/l/de/o/technical-product-owner-im-bereich-tax-technology-e-invoicing-mwd?source=Indeed
Компания,cbs Corporate Business Solutions
Локация,"Heidelberg, BW, DE"
Опубликовано,2026-06-11
Тип занятости,fulltime
Удалёнка,False
Email,christina.argyriadou@cbs-consulting.de
Сайт компании (board),https://de.indeed.com/cmp/CBS-Corporate-Business-Solutions
Сайт компании (прямой),https://www.cbs-consulting.com/en/about-cbs/company/
Адрес,Headquarter Germany Rudolf-Diesel-Straße 9 69115 Heidelberg


**📝 Описание:**

**Wir sind die Berater der Weltmarktführer: Hochmotivierte Experten, die sich miteinander vernetzen, um als erfolgreiches Team digitale End-to-End-Geschäftsprozesslösungen voranzutreiben. Gemeinsam stärken wir die Zukunft der beeindruckendsten Unternehmen der Welt aus Maschinen- und Anlagenbau, Automotive, Life Science, Pharma und Chemie, mit denen wir viel gemeinsam haben.**

Unser ausgezeichneter Ruf bei Hidden Champions öffnet Dir die Türen für anspruchsvolle Projekte mit einzigartigen Gestaltungsmöglichkeiten. Werde Teil einer der besten und verlässlichsten Beratungen und nutze vielfältige Entwicklungschancen. Als engagierter Mitarbeiter kannst Du bei uns schnell Verantwortung übernehmen und über Dich hinauswachsen. Du triffst auf Kollegen, die wirklich etwas bewegen wollen und als starkes Team zusammenhalten. cbs ist der Ort, an dem Experten bleiben wollen.

E-Invoicing und digitale Compliance verbreiten sich im Zuge der Digitalisierung rasend schnell. In viele Ländern ist e-Invoicing bereits verpflichtend, noch mehr werden in den kommenden Jahren folgen. Globale aufgestellte Unternehmen stehen vor der großen Herausforderung heterogene komplexe e-Invoicing Prozesse in vielen Ländern umzusetzen und betreiben zu müssen. cbs bietet im Bereich Managed Cloud Services ein einzigartiges Lösungsangebot um e-Invoicing auf globaler Ebene mit moderner Cloud-Architektur umzusetzen und zu betreiben. Die Kombination aus Managed Cloud Services und der Beratungsexpertise im Bereich Globalisierung, macht cbs im Bereich e-Invoicing zum führenden Dienstleister im SAP Umfeld in der DACH Region.

Zur Weiterentwicklung der cbs Managed Cloud Service suchen wir einen **Technical Product Owner im Bereich Tax Technology & E-Invoicing.**

### **Was dich erwartet**

* Fachliche und technische **Produktverantwortung** für Lösungen im Bereich **regulatorisches E‑Invoicing** (B2G/B2B), inkl. Anbindung an **staatliche Portale und Plattformen**
* Übersetzung **gesetzlicher und regulatorischer Anforderungen** (z. B. Formate, Meldepflichten, Länderbesonderheiten) in **klare Produkt‑ und Systemanforderungen**
* Unterstützen der **technischen Produktarchitektur** (Schnittstellen, APIs, Datenmodelle, Integrationsflüsse)
* **Implementierung** in agilen, cross‑funktionalen Teams (Scrum / Kanban)
* Enge Zusammenarbeit mit Architektur, Beratung, Regulatory Support und externen Partnern
* Sicherstellung von **Qualität, Sicherheit und Betriebskonzepten** in produktiven Systemen

### **Was wir uns wünschen**

* Erfahrung als (Technical) Product Owner, Business Analyst oder in einer vergleichbaren Schnittstellenrolle
* Fundierte Kenntnisse im Bereich **regulatorisches E‑Invoicing**  
  (z. B. staatliche Portale, Validierungsregeln, nationale und internationale Anforderungen)
* Praktische Erfahrung in der **Cloud-&Schnittstellenentwicklung** (z.B. Lobster\_data, Temporal, RabbitMQ)
* Starkes technisches Verständnis für **Systemarchitekturen, APIs (REST/SOAP), Schnittstellen, Datenbanken und Integrationsmuster**
* Fähigkeit, komplexe fachliche und technische Zusammenhänge **verständlich zu strukturieren und zu priorisieren**
* Erfahrung in der Mitarbeit in **Entwicklungsteams** und der Zusammenarbeit mit technischen Stakeholdern

* Verständnis für **Cloud‑, Sicherheits‑ und Skalierungsaspekte** ist ein Plus
* Strukturierte, lösungsorientierte Arbeitsweise mit Ownership‑Mentalität
* **Gute Deutschkenntnisse** sowie **sehr gute Englischkenntnisse** für die Arbeit im internationalen Umfeld

### **Unsere Benefits:**

**Aus- und Weiterbildung**

SAP- und Salesforce-Zertifizierungen, Competence Center-Tage, Fachspezifische Schulungen

**Betriebliche Gesundheitsförderung**

Zuschuss zur betrieblichen Altersvorsorge, Zusatzversicherungen, diverse Sportangebote wie Urban Sports Club, Yoga etc.

**Events**

Beratertage, Weihnachtsfeier, Sommerfest, Teamevents, Alumni-Treffen, Jubiläumsfeiern und -reisen

**Prämien**

Umsatzbeteiligungen, Prämienzahlungen für erfolgreich abgeschlossene Projekte, Attraktive Gehaltsmodelle, Mitarbeiterempfehlungsprämien

**Arbeitszeiten**

Flexible Arbeitszeitmodelle, mobiles Arbeiten, 30 Tage Urlaub, Workation

**Ausstattung**

Firmenhandy (iPhone), Notebook / Macbook, Mitarbeiter PC Programm, Ergonomische Arbeitsplätze, Kostenlose Getränke, Gemeinschaftsräume mit Wohlfühlcharakter

**Firmenwagen**

Firmenwagen inkl. Tankkarte/ Mobilitätspauschale, Bahncard, Jobrad

**Zuwendungen**

Zuschüsse für Kindergarten/Kita und außerordentliche Zuwendungen je Lebenssituation und Betriebszugehörigkeit

#### **Dein Ansprechpartner**

**Christina Argyriadou**

Senior Recruiterin

+49 15165577154

christina.argyriadou@cbs-consulting.de

Ansprechpartner: Christina Argyriadou

Berufserfahrung: Professionals

Bereich: Technology Consulting

Practice: Managed Cloud Services (MCS)

Ссылка (board),https://de.indeed.com/viewjob?jk=187fb3235143fbf5
Ссылка (прямая),https://cbsconsulting.recruitee.com/l/de/o/technical-product-owner-im-bereich-tax-technology-e-invoicing-mwd?source=Indeed
Компания,cbs Corporate Business Solutions
Локация,"Dortmund, NW, DE"
Опубликовано,2026-06-11
Тип занятости,fulltime
Удалёнка,False
Email,christina.argyriadou@cbs-consulting.de
Сайт компании (board),https://de.indeed.com/cmp/CBS-Corporate-Business-Solutions
Сайт компании (прямой),https://www.cbs-consulting.com/en/about-cbs/company/
Адрес,Headquarter Germany Rudolf-Diesel-Straße 9 69115 Heidelberg


**📝 Описание:**

**Wir sind die Berater der Weltmarktführer: Hochmotivierte Experten, die sich miteinander vernetzen, um als erfolgreiches Team digitale End-to-End-Geschäftsprozesslösungen voranzutreiben. Gemeinsam stärken wir die Zukunft der beeindruckendsten Unternehmen der Welt aus Maschinen- und Anlagenbau, Automotive, Life Science, Pharma und Chemie, mit denen wir viel gemeinsam haben.**

Unser ausgezeichneter Ruf bei Hidden Champions öffnet Dir die Türen für anspruchsvolle Projekte mit einzigartigen Gestaltungsmöglichkeiten. Werde Teil einer der besten und verlässlichsten Beratungen und nutze vielfältige Entwicklungschancen. Als engagierter Mitarbeiter kannst Du bei uns schnell Verantwortung übernehmen und über Dich hinauswachsen. Du triffst auf Kollegen, die wirklich etwas bewegen wollen und als starkes Team zusammenhalten. cbs ist der Ort, an dem Experten bleiben wollen.

E-Invoicing und digitale Compliance verbreiten sich im Zuge der Digitalisierung rasend schnell. In viele Ländern ist e-Invoicing bereits verpflichtend, noch mehr werden in den kommenden Jahren folgen. Globale aufgestellte Unternehmen stehen vor der großen Herausforderung heterogene komplexe e-Invoicing Prozesse in vielen Ländern umzusetzen und betreiben zu müssen. cbs bietet im Bereich Managed Cloud Services ein einzigartiges Lösungsangebot um e-Invoicing auf globaler Ebene mit moderner Cloud-Architektur umzusetzen und zu betreiben. Die Kombination aus Managed Cloud Services und der Beratungsexpertise im Bereich Globalisierung, macht cbs im Bereich e-Invoicing zum führenden Dienstleister im SAP Umfeld in der DACH Region.

Zur Weiterentwicklung der cbs Managed Cloud Service suchen wir einen **Technical Product Owner im Bereich Tax Technology & E-Invoicing.**

### **Was dich erwartet**

* Fachliche und technische **Produktverantwortung** für Lösungen im Bereich **regulatorisches E‑Invoicing** (B2G/B2B), inkl. Anbindung an **staatliche Portale und Plattformen**
* Übersetzung **gesetzlicher und regulatorischer Anforderungen** (z. B. Formate, Meldepflichten, Länderbesonderheiten) in **klare Produkt‑ und Systemanforderungen**
* Unterstützen der **technischen Produktarchitektur** (Schnittstellen, APIs, Datenmodelle, Integrationsflüsse)
* **Implementierung** in agilen, cross‑funktionalen Teams (Scrum / Kanban)
* Enge Zusammenarbeit mit Architektur, Beratung, Regulatory Support und externen Partnern
* Sicherstellung von **Qualität, Sicherheit und Betriebskonzepten** in produktiven Systemen

### **Was wir uns wünschen**

* Erfahrung als (Technical) Product Owner, Business Analyst oder in einer vergleichbaren Schnittstellenrolle
* Fundierte Kenntnisse im Bereich **regulatorisches E‑Invoicing**  
  (z. B. staatliche Portale, Validierungsregeln, nationale und internationale Anforderungen)
* Praktische Erfahrung in der **Cloud-&Schnittstellenentwicklung** (z.B. Lobster\_data, Temporal, RabbitMQ)
* Starkes technisches Verständnis für **Systemarchitekturen, APIs (REST/SOAP), Schnittstellen, Datenbanken und Integrationsmuster**
* Fähigkeit, komplexe fachliche und technische Zusammenhänge **verständlich zu strukturieren und zu priorisieren**
* Erfahrung in der Mitarbeit in **Entwicklungsteams** und der Zusammenarbeit mit technischen Stakeholdern

* Verständnis für **Cloud‑, Sicherheits‑ und Skalierungsaspekte** ist ein Plus
* Strukturierte, lösungsorientierte Arbeitsweise mit Ownership‑Mentalität
* **Gute Deutschkenntnisse** sowie **sehr gute Englischkenntnisse** für die Arbeit im internationalen Umfeld

### **Unsere Benefits:**

**Aus- und Weiterbildung**

SAP- und Salesforce-Zertifizierungen, Competence Center-Tage, Fachspezifische Schulungen

**Betriebliche Gesundheitsförderung**

Zuschuss zur betrieblichen Altersvorsorge, Zusatzversicherungen, diverse Sportangebote wie Urban Sports Club, Yoga etc.

**Events**

Beratertage, Weihnachtsfeier, Sommerfest, Teamevents, Alumni-Treffen, Jubiläumsfeiern und -reisen

**Prämien**

Umsatzbeteiligungen, Prämienzahlungen für erfolgreich abgeschlossene Projekte, Attraktive Gehaltsmodelle, Mitarbeiterempfehlungsprämien

**Arbeitszeiten**

Flexible Arbeitszeitmodelle, mobiles Arbeiten, 30 Tage Urlaub, Workation

**Ausstattung**

Firmenhandy (iPhone), Notebook / Macbook, Mitarbeiter PC Programm, Ergonomische Arbeitsplätze, Kostenlose Getränke, Gemeinschaftsräume mit Wohlfühlcharakter

**Firmenwagen**

Firmenwagen inkl. Tankkarte/ Mobilitätspauschale, Bahncard, Jobrad

**Zuwendungen**

Zuschüsse für Kindergarten/Kita und außerordentliche Zuwendungen je Lebenssituation und Betriebszugehörigkeit

#### **Dein Ansprechpartner**

**Christina Argyriadou**

Senior Recruiterin

+49 15165577154

christina.argyriadou@cbs-consulting.de

Ansprechpartner: Christina Argyriadou

Berufserfahrung: Professionals

Bereich: Technology Consulting

Practice: Managed Cloud Services (MCS)

## 5. Открыть конкретную вакансию по индексу

Поменяй `i` на номер строки из таблицы обзора (раздел 3).


In [7]:
i = 0
show_jobs(jobs.iloc[[i]], n=1)


Ссылка (board),https://de.indeed.com/viewjob?jk=a9f0797444582dcb
Ссылка (прямая),https://www.aareal-bank.com/karriere/jobs-bewerbung/berufserfahren?tx_aarealjobs_hrportallistshow%5Baction%5D=show&tx_aarealjobs_hrportallistshow%5Bcontroller%5D=Job&tx_aarealjobs_hrportallistshow%5Bjob%5D=732&cHash=15afec34b03d71eb85f6d0f32324f5f5
Компания,Aareal Bank
Локация,"Wiesbaden, HE, DE"
Опубликовано,2026-06-11
Удалёнка,False
Сайт компании (board),https://de.indeed.com/cmp/Aareal-Bank
Сайт компании (прямой),https://www.aareal-bank.com
Адрес,"Wiesbaden, GM"
Сотрудников,"1,001 to 5,000"
Выручка,$500M to $1B (USD)


**📝 Описание:**

Für diese Position suchen wir eine ideenreiche und innovative Persönlichkeit, die dank ihrer exzellenten Analysefähigkeiten das Team mit Fokus auf Bilanzierungslösungen verstärkt. Da wir hierbei weitgehend auf unsere SAP-basierten Eigenentwicklungen zurückgreifen, sind keine spezifischen SAP-Modulkenntnisse erforderlich. Wenn du hingegen über einen ausgeprägten Teamgeist verfügst und den Dingen gerne durch konstruktives und kritisches Nachfragen auf den Grund gehst, sagen wir: Herzlich willkommen bei der Aareal Bank Group.

#### **Darauf kannst du dich freuen:**

* Als Junior ABAP Entwickler (w/m/d) verstärkst Du das Team rund um die Weiterentwicklung und Wartung einer komplexen Bilanzierungslösung unter SAP S/4HANA – eine Eigenentwicklung, die tief in die Finanzprozesse der Bank integriert ist.
* Die Aareal Bank bietet ihren Kunden strukturierte Finanzierungen. Die bilanzielle Abbildung solcher Geschäfte ist rundum anspruchsvoll und sorgt sicher dafür, dass keine Langeweile aufkommt.
* Intensive Einarbeitung durch ein Team von erfahrenen Kolleginnen und Kollegen in alle relevanten Aspekte der Bilanzierung, dazu gehören insbesondere die Klassifizierung und Bewertung von Finanzinstrumenten, Bildung der Risikovorsorge sowie Hedge Accounting.
* Deine Einarbeitung wird individuell gestaltet und erfolgt in einem interdisziplinären Team mit Entwicklern, Business-Analysten und Fachbereichen. Sie umfasst sowohl die konkrete technische Implementierung als auch die Vermittlung fachlicher Grundlagen zu den geltenden Rechnungslegungsvorschriften.
* Über die reinen Wartungs- und Entwicklungsarbeiten hinaus erstellst Du auch technische Konzepte sowie Betriebslösungen, die Stabilität, Performance und Wartbarkeit sicherstellen.

#### **Darauf können wir uns freuen:**

* Ein abgeschlossenes Studium der Informatik, Wirtschaftsinformatik, Mathematik, Wirtschaftsmathematik oder Wirtschaftswissenschaften mit technischem Schwerpunkt - oder eine vergleichbare Qualifikation
* Nachweisbar erfahren im Umgang mit SAP ABAP OO und ABAP 4HANA. Hilfreich ist darüber hinaus Erfahrung in CDS-Views und OData.
* Im Rahmen einer Praktikums- oder Werkstudententätigkeit hast du erste Erfahrung in Unternehmen in der Wartung und Entwicklung im SAP-Umfeld gesammelt. Natürlich bist Du auch mit darüber hinausgehender Erfahrung bei uns willkommen.
* Idealerweise Routine mit agilen Vorgehensmodellen – z. B. SCRUM oder Kanban
* Nice to have: Kenntnisse im Finanzwesen und Bilanzierung sowie Modulkenntnisse in SAP FI/CO
* Fließende Deutschkenntnisse in Wort und Schrift

**Das machen wir aanders!**
---------------------------

* ### **Aabwechslungsreiche Aufgaben**

  Bei uns kannst du deine Expertise einbringen, Themen voranbringen und Neues gestalten. Wir haben eine Kultur, in der sich alle aktiv einbringen und Einfluss nehmen können.
* ### **Aaußergewöhnliche Weiterentwicklung**

  Wir fördern jede Person gleichermaßen, deshalb unterstützen wir dich mit individuellen Entwicklungsmöglichkeiten und Lernwegen. Damit du bestehende Kompetenzen ausbauen und neue zukunftsweisende Fähigkeiten aufbauen kannst.
* ### **Aausgeglichene Work-Life Balance**

  Mit mobilem Arbeiten und flexiblen Arbeitszeitmodellen hast du die Möglichkeit, deine Arbeit individuell zu gestalten. So lassen sich Arbeit, Familie und die persönliche Lebenssituation bestmöglich unter einen Hut bringen.
* ### **Aausgezeichnete Benefits**

  Unsere Services und Benefits bieten dir einen echten Mehrwert, die uns so zu einem Top-Arbeitgeber machen. Bei uns erhältst du beispielsweise Vergünstigungen bei Kooperationspartnern oder Zuschüsse zum Nahverkehr. Ebenso sorgen wir mit einer betrieblichen Altersversorgung oder vermögenswirksamen Leistungen für deine Zukunft vor.
* ### **Aaktives Gesundheitsmanagement**

  Ärztliche Check-ups, Beratungsangebote, gemeinsame Sportevents, unsere beliebte Kantine oder Kaffeebar – Bei uns kannst du das nutzen, was deine Gesundheit fördert und dir guttut.

**Willkommen in der Welt der Aareal Bank!**
-------------------------------------------

Wir sind Mittelständler und Global Player. Als internationaler Immobilienspezialist arbeiten wir alle auf ein gemeinsames Ziel hin: den Erfolg unserer Kunden. Bei uns arbeiten über 1.000 Menschen aus 44 Nationen auf drei Kontinenten daran, dass wir als Bank ganz schön aanders sind! Wir sind verantwortungsvoller und erstklassiger Ausbildungsbetrieb und eine Organisation mit viel Expertise und kurzen Wegen, viel Gestaltungsspielraum und dem Anspruch, für unsere Kunden immer besser zu werden. Das macht uns stark und vor allem: aanders!

## 6. Фильтры и поиск по результатам (pandas)

Всё это — обычный DataFrame, можно фильтровать как угодно.


In [8]:
# Примеры фильтрации
df = jobs.copy()

# а) строго по городу
city = "Heidelberg"
in_city = df[df["location"].fillna("").str.contains(city, case=False)]
print(f"В '{city}': {len(in_city)}")

# б) по тайтлу: оставить нужное, выкинуть мусор
keep = df["title"].str.contains("analyst|analytics", case=False, na=False)
drop = df["title"].str.contains("praktikum|werkstudent|studium|ausbildung", case=False, na=False)
clean = df[keep & ~drop]
print(f"После чистки тайтлов: {len(clean)}")

# в) только с указанной зарплатой
with_salary = df[df["min_amount"].notna()]
print(f"С зарплатой: {len(with_salary)}")

# показать отфильтрованное:
# show_jobs(clean, n=5)
clean[["site","title","company","location","date_posted","job_url"]].head(20)


В 'Heidelberg': 1
После чистки тайтлов: 26
С зарплатой: 0


,site,title,company,location,date_posted,job_url
1,indeed,Data Analyst als Experte Fachkonzepte & Data Lake (m/w/d),Zurich Insurance,"Frankfurt am Main, HE, DE",2026-06-11,https://de.indeed.com/viewjob?jk=a893707c9d4a9f8f
5,indeed,Business Analyst Legal Operations (m/w/d),REWE Group,"Köln, NW, DE",2026-06-11,https://de.indeed.com/viewjob?jk=8af2b590ed5c32bc
6,indeed,Salesforce AI & Analytics Analyst (all genders),Accenture,"Düsseldorf, NW, DE",2026-06-11,https://de.indeed.com/viewjob?jk=2487537631233563
7,indeed,Salesforce AI & Analytics Analyst (all genders),Accenture,"Stuttgart, BW, DE",2026-06-11,https://de.indeed.com/viewjob?jk=97f0444523bcd251
8,indeed,Salesforce AI & Analytics Analyst (all genders),Accenture,"München, BY, DE",2026-06-11,https://de.indeed.com/viewjob?jk=bd4b31c8d9bf7c6b
9,indeed,Salesforce AI & Analytics Analyst (all genders),Accenture,"Kronberg, HE, DE",2026-06-11,https://de.indeed.com/viewjob?jk=b6a2f2b6569ff293
10,indeed,Sales Analyst (m/f/x),Magna International,"München, BY, DE",2026-06-11,https://de.indeed.com/viewjob?jk=761ea4c77678c4c1
11,indeed,Sales Analyst (m/f/x),Magna International,"Sailauf, BY, DE",2026-06-11,https://de.indeed.com/viewjob?jk=115f62386b5766e4
12,indeed,Finance Business Analyst (f/m/d),Siemens Energy,"Kirchheim unter Teck, BW, DE",2026-06-11,https://de.indeed.com/viewjob?jk=8925d77447675a5d
13,indeed,Business Analyst / Requirements Engineer (m/w/d)*,ALH Gruppe,"Stuttgart, BW, DE",2026-06-11,https://de.indeed.com/viewjob?jk=c0c47aac95cb5cde


## 7. Сохранить результаты (CSV / Excel / JSON)

In [9]:
import csv as _csv
from datetime import datetime
stamp = datetime.now().strftime("%Y-%m-%d_%H%M")
base = f"jobs_{stamp}"

jobs.to_csv(f"{base}.csv", quoting=_csv.QUOTE_NONNUMERIC, escapechar="\\", index=False)
jobs.to_json(f"{base}.json", orient="records", date_format="iso", force_ascii=False, indent=2)
try:
    jobs.to_excel(f"{base}.xlsx", index=False)
    print("Сохранено:", f"{base}.csv,", f"{base}.xlsx,", f"{base}.json")
except Exception as e:
    print("CSV+JSON сохранены; Excel пропущен:", e)


Сохранено: jobs_2026-06-11_2035.csv, jobs_2026-06-11_2035.xlsx, jobs_2026-06-11_2035.json


---
### Шпаргалка по параметрам `CONFIG`

| Параметр | Назначение |
|---|---|
| `site_name` | площадки: linkedin, indeed, glassdoor, google, zip_recruiter, bayt, naukri, bdjobs |
| `search_term` | ключевой запрос; для Indeed работают операторы `"точная фраза"`, `-минус`, `OR` |
| `google_search_term` | отдельный запрос только для google |
| `location` / `distance` | город/страна и радиус (мили) |
| `country_indeed` | страна для Indeed/Glassdoor (обязательно) |
| `is_remote` / `job_type` / `easy_apply` | фильтры занятости |
| `results_wanted` / `hours_old` / `offset` | объём, свежесть, сдвиг |
| `linkedin_fetch_description` | полные описания LinkedIn (медленнее) |
| `linkedin_company_ids` | поиск по конкретным компаниям LinkedIn |
| `proxies` / `ca_cert` / `user_agent` | сеть (обход блокировок) |
| `description_format` | markdown / html / plain |
| `enforce_annual_salary` | привести зарплаты к годовым |

Тот же функционал из терминала: `python jobspy_cli.py --help`
